# Can we predict a warranty claim within a year?
Follows `protocol.yaml`: fixed holdout (`holdout.csv`), 5-fold stratified CV on training rows, choice by mean CV ROC AUC, holdout scored once.

All features (channel, device age, price, daily hours) are recorded at sale/registration, so are known at prediction time. `device_serial` is excluded (identifier).

In [ ]:
import json, hashlib
import numpy as np, pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.metrics import roc_auc_score

assert hashlib.sha256(open('data.csv','rb').read()).hexdigest()=='25552b73fd810f7dfa81a14849736291d84f91da2f8b5a58bcfaf0cc018962f4'
df = pd.read_csv('data.csv')
hold_idx = pd.read_csv('holdout.csv')['row'].to_numpy()   # 0-based positions in data.csv
assert len(set(hold_idx))==1400 and hold_idx.min()>=0 and hold_idx.max()<len(df)
mask = np.zeros(len(df),bool); mask[hold_idx]=True
train, hold = df[~mask], df[mask]
train_rows = np.flatnonzero(~mask)
target='claim_within_year'
features=['channel','device_age_years','price_eur','daily_hours']
Xtr, ytr = train[features], train[target]
Xho, yho = hold[features], hold[target]
print(len(train), len(hold), ytr.mean(), yho.mean())

In [ ]:
print(train.describe(include='all').T)
print(train.groupby('channel')[target].agg(['mean','size']))
for f in ['device_age_years','price_eur','daily_hours']:
    print(f, train.groupby(pd.qcut(train[f],5,duplicates='drop'),observed=True)[target].mean().round(3).to_dict())

## Candidates (named before any scoring)
1. `baseline_prior` – predicts class prior (trivial)
2. `logreg` – scaled numeric + one-hot channel
3. `logreg_C0.1` – stronger regularisation
4. `random_forest` (min_samples_leaf=20)
5. `hist_gb` (shallow, lr 0.05)

Preprocessing lives inside the pipelines, so it is fitted within each CV fold.

In [ ]:
num=['device_age_years','price_eur','daily_hours']; cat=['channel']
def prep(): return ColumnTransformer([('n',StandardScaler(),num),('c',OneHotEncoder(handle_unknown='ignore'),cat)])
candidates = {
 'baseline_prior': make_pipeline(prep(), DummyClassifier(strategy='prior')),
 'logreg': make_pipeline(prep(), LogisticRegression(max_iter=1000)),
 'logreg_C0.1': make_pipeline(prep(), LogisticRegression(C=0.1, max_iter=1000)),
 'random_forest': make_pipeline(prep(), RandomForestClassifier(n_estimators=300, min_samples_leaf=20, random_state=0)),
 'hist_gb': make_pipeline(prep(), HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=200, random_state=0)),
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
cv_scores = {k: cross_val_score(m, Xtr, ytr, cv=cv, scoring='roc_auc').tolist() for k,m in candidates.items()}
summary = pd.DataFrame({k:{'mean':np.mean(v),'std':np.std(v)} for k,v in cv_scores.items()}).T
print(summary.sort_values('mean',ascending=False))
chosen = summary['mean'].idxmax(); print('chosen:', chosen)

## Holdout: chosen candidate refitted on all training rows, scored once

In [ ]:
final = candidates[chosen].fit(Xtr, ytr)
holdout_auc = roc_auc_score(yho, final.predict_proba(Xho)[:,1])
print('holdout roc_auc', holdout_auc)

In [ ]:
folds=[[train_rows[tr].tolist(),train_rows[te].tolist()] for tr,te in cv.split(Xtr,ytr)]  # [train, test] row positions per fold
evidence={
 'features':features,
 'candidates':list(candidates),
 'cv_folds':folds,
 'cv_scores':cv_scores,
 'chosen':chosen,
 'holdout_rows':sorted(int(i) for i in hold_idx),
 'holdout_scores':{chosen: holdout_auc},
}
json.dump(evidence, open('evidence.json','w'))

## Conclusion
The printed CV table and holdout AUC above are the result; the holdout AUC is the headline. An AUC of 0.5 would mean no predictability.